# Recommender System

- A recommender system for Anime. I will use collaborative filtering for it.

In [1]:
# Importing Libraries
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gc
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
warnings.simplefilter(action='ignore', category=FutureWarning)

In [30]:
anime = pd.read_csv("C:/Users/BVCaridad/Documents/Project Portfolio/Recommender System - Anime/anime.csv")
rating = pd.read_csv("C:/Users/BVCaridad/Documents/Project Portfolio/Recommender System - Anime/rating.csv")
gc.collect()

958

In [32]:
anime = anime.head(1000)
trimmed_rating = rating[rating['anime_id'].isin(anime['anime_id'])] 
unique_user_id = trimmed_rating['user_id'].unique()[:10000]

rating = trimmed_rating[trimmed_rating['user_id'].isin(unique_user_id)]

In [8]:
rating.head()

,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


In [40]:
n_ratings = len(rating)
n_animes = len(rating['anime_id'].unique())
n_users = len(rating['user_id'].unique())

print(f"Number of ratings: {n_ratings}")
print(f"Number of unique anime's: {n_animes}")
print(f"Number of unique users: {n_users}")
print(f"Average ratings per user: {round(n_ratings/n_users, 2)}")
print(f"Average ratings per anime: {round(n_ratings/n_animes, 2)}")


Number of ratings: 495852
Number of unique anime's: 976
Number of unique users: 10000
Average ratings per user: 49.59
Average ratings per anime: 508.05


In [41]:
user_freq = rating[['user_id', 'anime_id']].groupby(
    'user_id').count().reset_index()
user_freq.columns = ['user_id', 'n_ratings']
print(user_freq.head())

   user_id  n_ratings
0        1         47
1        2          2
2        3         47
3        4         26
4        5        180


In [42]:
# Find Lowest and Highest rated anime:
mean_rating = rating.groupby('anime_id')[['rating']].mean()
# Lowest rated anime
lowest_rated = mean_rating['rating'].idxmin()
anime.loc[anime['anime_id'] == lowest_rated]
# Highest rated anime
highest_rated = mean_rating['rating'].idxmax()
anime.loc[anime['anime_id'] == highest_rated]
# show number of people who rated anime rated anime highest
rating[rating['anime_id']==highest_rated]
# show number of people who rated anime rated anime lowest
rating[rating['anime_id']==lowest_rated]

## the above anime has very low dataset. We will use bayesian average
anime_stats = rating.groupby('anime_id')[['rating']].agg(['count', 'mean'])
anime_stats.columns = anime_stats.columns.droplevel()


# First Version

- Uses a user-item matrix with ratings for item-based collaborative filtering to find similar anime
- Recommends similar anime based on a user's highest-rated anime

In [43]:
def create_matrix(df):
    
    N = len(df['user_id'].unique())
    M = len(df['anime_id'].unique())
    
    # Map Ids to indices
    user_mapper = dict(zip(np.unique(df["user_id"]), list(range(N))))
    anime_mapper = dict(zip(np.unique(df["anime_id"]), list(range(M))))
    
    # Map indices to IDs
    user_inv_mapper = dict(zip(list(range(N)), np.unique(df["user_id"])))
    anime_inv_mapper = dict(zip(list(range(M)), np.unique(df["anime_id"])))
    
    user_index = [user_mapper[i] for i in df['user_id']]
    anime_index = [anime_mapper[i] for i in df['anime_id']]

    X = csr_matrix((df["rating"], (anime_index, user_index)), shape=(M, N))
    
    return X, user_mapper, anime_mapper, user_inv_mapper, anime_inv_mapper
    
X, user_mapper, anime_mapper, user_inv_mapper, anime_inv_mapper = create_matrix(rating)


In [44]:
"""
Find similar anime using KNN
"""
def find_similar_anime(anime_id, X, k, metric='cosine', show_distance=False):
    
    neighbour_ids = []
    
    anime_ind = anime_mapper[anime_id]
    anime_vec = X[anime_ind]
    k+=1
    kNN = NearestNeighbors(n_neighbors=k, algorithm="brute", metric=metric)
    kNN.fit(X)
    anime_vec = anime_vec.reshape(1,-1)
    neighbour = kNN.kneighbors(anime_vec, return_distance=show_distance)
    for i in range(0,k):
        n = neighbour.item(i)
        neighbour_ids.append(anime_inv_mapper[n])
    neighbour_ids.pop(0)
    return neighbour_ids


anime_titles = dict(zip(anime['anime_id'], anime['name']))

anime_id = 32281

similar_ids = find_similar_anime(anime_id, X, k=10)
anime_title = anime_titles[anime_id]

print(f"Since you watched {anime_title}")
for i in similar_ids:
    print(anime_titles[i])


Since you watched Kimi no Na wa.
Boku dake ga Inai Machi
ReLIFE
Shigatsu wa Kimi no Uso
Re:Zero kara Hajimeru Isekai Seikatsu
Kokoro ga Sakebitagatterunda.
Hai to Gensou no Grimgar
Orange
Charlotte
Noragami Aragoto
One Punch Man


In [45]:
def recommend_animes_for_user(user_id, X, user_mapper, anime_mapper, anime_inv_mapper, k=10):
    df1 = rating[rating['user_id'] == user_id]
    
    if df1.empty:
        print(f"User with ID {user_id} does not exist.")
        return

    anime_id = df1[df1['rating'] == max(df1['rating'])]['anime_id'].iloc[0]

    anime_titles = dict(zip(anime['anime_id'], anime['name']))

    similar_ids = find_similar_anime(anime_id, X, k)
    anime_title = anime_titles.get(anime_id, "anime not found")

    if anime_title == "anime not found":
        print(f"anime with ID {anime_id} not found.")
        return

    print(f"Since you watched {anime_title}, you might also like:")
    for i in similar_ids:
        print(anime_titles.get(i, "anime not found"))


In [46]:
user_id = 1  # Replace with the desired user ID
recommend_animes_for_user(user_id, X, user_mapper, anime_mapper, anime_inv_mapper, k=10)


Since you watched Sword Art Online, you might also like:
Shingeki no Kyojin
No Game No Life
Angel Beats!
Mirai Nikki (TV)
Ao no Exorcist
Guilty Crown
Tokyo Ghoul
Akame ga Kill!
Noragami
Log Horizon


# Second Version

- Creates a watch matrix indicating whether a user has watched an anime
- Finds similar users using a user-based collaborative filtering
- Recommends anime based on what similar users have watched

- Recommends the top 3 animes

In [47]:
def create_watch_matrix(df):
    N = len(df['user_id'].unique())
    M = len(df['anime_id'].unique())
    
    user_mapper = dict(zip(np.unique(df["user_id"]), list(range(N))))
    anime_mapper = dict(zip(np.unique(df["anime_id"]), list(range(M))))
    
    user_inv_mapper = dict(zip(list(range(N)), np.unique(df["user_id"])))
    anime_inv_mapper = dict(zip(list(range(M)), np.unique(df["anime_id"])))
    
    user_index = [user_mapper[i] for i in df['user_id']]
    anime_index = [anime_mapper[i] for i in df['anime_id']]
    
    watch_matrix = csr_matrix((np.ones(len(df)), (user_index, anime_index)), shape=(N, M))
    
    return watch_matrix, user_mapper, anime_mapper, user_inv_mapper, anime_inv_mapper

watch_matrix, user_mapper, anime_mapper, user_inv_mapper, anime_inv_mapper = create_watch_matrix(rating)

In [48]:
def find_similar_users(user_id, watch_matrix, k, metric='cosine'):
    neighbour_ids = []
    
    if user_id not in user_mapper:
        print(f"User ID {user_id} not found in user_mapper.")
        return []
    
    user_ind = user_mapper[user_id]
    user_vec = watch_matrix[user_ind]
    k += 1
    kNN = NearestNeighbors(n_neighbors=k, algorithm="brute", metric=metric)
    kNN.fit(watch_matrix)
    user_vec = user_vec.reshape(1, -1)
    neighbour = kNN.kneighbors(user_vec, return_distance=False)
    
    for i in range(1, k):  # Start from 1 to skip the first neighbour (itself)
        n = neighbour.item(i)
        neighbour_ids.append(user_inv_mapper[n])
    
    return neighbour_ids

In [49]:
def recommend_animes_for_all_users(watch_matrix, user_mapper, anime_mapper, anime_inv_mapper, k=3):
    recommendations = []

    for user_id in rating['user_id'].unique():
        similar_users = find_similar_users(user_id, watch_matrix, k=10)
        
        if not similar_users:
            continue
        
        watched_animes = set(rating[rating['user_id'] == user_id]['anime_id'])
        recommended_animes = set()
        
        for similar_user in similar_users:
            similar_user_animes = set(rating[rating['user_id'] == similar_user]['anime_id'])
            recommended_animes.update(similar_user_animes - watched_animes)
        
        recommended_animes = list(recommended_animes)
        recommended_animes = recommended_animes[:k]
        recommended_anime_titles = [anime_titles.get(anime_id, "anime not found") for anime_id in recommended_animes]
        
        recommendations.append((user_id, recommended_anime_titles))

    return recommendations

user_recommendations = recommend_animes_for_all_users(watch_matrix, user_mapper, anime_mapper, anime_inv_mapper, k=3)

# Convert recommendations to DataFrame
recommendations_df = pd.DataFrame(user_recommendations, columns=['user_id', 'recommendations'])

In [50]:
recommendations_df

,user_id,recommendations
0,1,"[Soul Eater, Re:Zero kara Hajimeru Isekai Seik..."
1,2,"[Another, Haikyuu!!, Kyoukai no Kanata]"
2,3,"[Cowboy Bebop, Yamada-kun to 7-nin no Majo (TV..."
3,4,"[Cowboy Bebop, Ano Hi Mita Hana no Namae wo Bo..."
4,5,"[Cowboy Bebop, Kyoukai no Kanata Movie: I&#039..."
...,...,...
9995,10193,"[Toradora!, Soul Eater, Trigun]"
9996,10194,"[Cowboy Bebop, Cowboy Bebop: Tengoku no Tobira..."
9997,10195,"[Majo no Takkyuubin, Cowboy Bebop, Tenkuu no S..."
9998,10196,"[Majo no Takkyuubin, Soul Eater, Yamada-kun to..."
